In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from utils import preprocess_data, get_features_and_target
from regression_utils import (
    train_and_compare_regression,
    print_results_table
)

## 1. ПОДГОТОВКА ДАННЫХ

In [3]:
#Подготовка данных
df = preprocess_data('data.xlsx')
X, y = get_features_and_target(df, 'SI', task_type='regression')

print(f"\nЦелевая переменная: SI (логарифмированная через log1p)")
print(f"Размер X: {X.shape}, размер y: {y.shape}")
print(f"\nСтатистика y (после log1p):")
print(f"  min:    {y.min():.3f}")
print(f"  max:    {y.max():.3f}")
print(f"  mean:   {y.mean():.3f}")
print(f"  median: {y.median():.3f}")
print(f"  std:    {y.std():.3f}")

Загружено: 1001 объектов, 213 столбцов
Удалено константных признаков: 33
Пропуски заполнены медианой каждого столбца
Удалено признаков с корреляцией > 0.95: 33
Итого: 1001 объектов, 147 столбцов (144 признаков)

Целевая переменная: SI (логарифмированная через log1p)
Размер X: (1001, 144), размер y: (1001,)

Статистика y (после log1p):
  min:    0.011
  max:    9.656
  mean:   2.042
  median: 1.578
  std:    1.456


In [4]:
# Запускаем обучение всех моделей с подбором гиперпараметров
results_df, best_models, splits = train_and_compare_regression(
    X, y,
    target_name='SI',
    cv=5,
    test_size=0.3,
    random_state=42
)

X_train, X_test, y_train, y_test = splits

In [5]:
#Вывод итоговой таблицы

print_results_table(results_df, target_name='SI')

from pathlib import Path; Path('results').mkdir(parents=True, exist_ok=True)
results_df.to_csv('results/results_regression_SI.csv', index=False)


Итоговая таблица сравнений для SI
           Model     CV R2  Train R2   Test R2  Test MAE  Test RMSE
    RandomForest  0.292560  0.719983  0.288955  0.952623   1.287718
             SVR  0.232367  0.471004  0.263880  0.928331   1.310227
GradientBoosting  0.261077  0.651696  0.262354  0.972823   1.311585
             KNN  0.210879  0.416932  0.261077  0.987401   1.312719
    DecisionTree  0.136788  0.409518  0.257279  0.987107   1.316089
           Lasso  0.128405  0.320862  0.219313  1.000317   1.349307
           Ridge  0.149757  0.315708  0.217244  1.001329   1.351093
LinearRegression -0.318230  0.443897 -0.065152  1.144627   1.576079

 Лучшая модель: RandomForest (Test R2 = 0.2890) ***


In [6]:
# Сравнение моделей
plt.figure(figsize=(12, 6))
sorted_df = results_df.sort_values('Test R2', ascending=True)
plt.barh(sorted_df['Model'], sorted_df['Test R2'], color='steelblue', edgecolor='black')
plt.xlabel('R^2 на тестовой выборке')
plt.title('Сравнение моделей регрессии по R^2 (SI)')
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
plt.tight_layout()
plt.savefig('results/comparison_SI.png', dpi=100, bbox_inches='tight')
plt.close()

# Предсказания vs реальные
best_model_name = results_df.iloc[0]['Model']
best_model = best_models[best_model_name]
y_pred_test = best_model.predict(X_test)

plt.figure(figsize=(8, 8))
plt.scatter(y_test, y_pred_test, alpha=0.5, color='steelblue')
min_val = min(y_test.min(), y_pred_test.min())
max_val = max(y_test.max(), y_pred_test.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', label='Идеальное предсказание')
plt.xlabel('Истинное log(1 + SI)')
plt.ylabel('Предсказанное log(1 + SI)')
plt.title(f'Предсказания vs реальные значения\n(лучшая модель: {best_model_name})')
plt.legend()
plt.tight_layout()
plt.savefig('results/predictions_SI.png', dpi=100, bbox_inches='tight')
plt.close()


In [7]:
final_model = best_model.named_steps['model']
if hasattr(final_model, 'feature_importances_'):
    print(f"\nТоп-15 важных признаков для модели {best_model_name} ")
    importances = pd.Series(final_model.feature_importances_, index=X.columns)
    top_features = importances.sort_values(ascending=False).head(15)
    for feat, imp in top_features.items():
        print(f"  {feat:30s}: {imp:.4f}")

    plt.figure(figsize=(10, 6))
    top_features.sort_values().plot(kind='barh', color='coral', edgecolor='black')
    plt.xlabel('Важность признака')
    plt.title(f'Топ-15 признаков для предсказания SI\n({best_model_name})')
    plt.tight_layout()
    plt.savefig('results/feature_importance_SI.png', dpi=100, bbox_inches='tight')
    plt.close()


Топ-15 важных признаков для модели RandomForest 
  VSA_EState6                   : 0.1004
  VSA_EState8                   : 0.0524
  BCUT2D_CHGLO                  : 0.0398
  qed                           : 0.0343
  SMR_VSA7                      : 0.0303
  BCUT2D_MRLOW                  : 0.0264
  RingCount                     : 0.0243
  Ipc                           : 0.0230
  BCUT2D_LOGPHI                 : 0.0226
  BCUT2D_LOGPLOW                : 0.0195
  VSA_EState2                   : 0.0179
  BCUT2D_MRHI                   : 0.0166
  Kappa3                        : 0.0164
  BCUT2D_CHGHI                  : 0.0158
  BCUT2D_MWLOW                  : 0.0154
